# Clase 020 — Álgebra lineal

**Parte 0** · `numpy.linalg`.

> 🎯 Operar con vectores y matrices al nivel necesario para ML. Por qué NUNCA usar `inv`.

> ⏱️ ~90 min

## ⚙️ Setup

In [ ]:
import numpy as np
import time
rng = np.random.default_rng(42)

## 1️⃣ El operador `@` (PEP 465)

Desde Python 3.5, `@` es el operador estándar para **multiplicación matricial** (no elementwise, que es `*`).

```python
C = A @ B           # multiplicación matricial
C = A.dot(B)        # equivalente, sintaxis vieja
C = np.matmul(A, B) # equivalente, función explícita
C = A * B           # ¡elementwise! distinto
```

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print('A @ B (matricial):')
print(A @ B)
print('\nA * B (elementwise):')
print(A * B)

## 2️⃣ Producto punto vector·vector

In [ ]:
a = rng.normal(0, 1, 100)
b = rng.normal(0, 1, 100)

print(f'a @ b        : {a @ b:.4f}')
print(f'np.dot(a, b) : {np.dot(a, b):.4f}')
print(f'sum(a*b)     : {(a*b).sum():.4f}  ← lo mismo')

## 3️⃣ Resolver `Ax = b`: `solve` vs `inv`

**REGLA**: para resolver `Ax = b`, usa `np.linalg.solve(A, b)`, **NUNCA** `np.linalg.inv(A) @ b`.

**Por qué**:
- `inv` calcula la inversa completa (O(n³) caro)
- `solve` usa descomposición LU (O(n³) pero con constante menor)
- `inv` es **numéricamente inestable** (amplifica errores)
- `solve` no construye la inversa, evita ese error

In [ ]:
N = 500
A = rng.normal(0, 1, (N, N))
b = rng.normal(0, 1, N)

t0 = time.perf_counter(); x_inv = np.linalg.inv(A) @ b; t1 = time.perf_counter()
t2 = time.perf_counter(); x_solve = np.linalg.solve(A, b); t3 = time.perf_counter()

print(f'inv(A) @ b : {(t1-t0)*1000:.1f} ms')
print(f'solve(A,b) : {(t3-t2)*1000:.1f} ms')
print(f'speedup    : {(t1-t0)/(t3-t2):.1f}×')

# Precisión: residual ||Ax - b||
print(f'\nresidual inv  : {np.linalg.norm(A @ x_inv - b):.2e}')
print(f'residual solve: {np.linalg.norm(A @ x_solve - b):.2e}')

## 4️⃣ Diagnóstico estructural

```python
np.linalg.norm(v)         # norma L2
np.linalg.det(A)          # determinante (cuidado: 0 ⇒ singular)
np.linalg.matrix_rank(A)  # rango
np.trace(A)               # traza (suma diagonal)
np.linalg.cond(A)         # número de condición (estabilidad)
```

**`cond(A)` grande (>1e10) ⇒ matriz mal condicionada ⇒ `solve` perderá precisión**.

In [ ]:
v = np.array([3, 4])
print(f'norma L2 [3,4]: {np.linalg.norm(v)}  (= 5)')

M = np.array([[2, 1], [1, 3]])
print(f'\nM = {M.tolist()}')
print(f'det      : {np.linalg.det(M):.4f}')
print(f'rank     : {np.linalg.matrix_rank(M)}')
print(f'trace    : {np.trace(M)}')
print(f'cond     : {np.linalg.cond(M):.2f}')

# Matriz singular
S = np.array([[1, 2], [2, 4]])
print(f'\nMatriz singular:')
print(f'det      : {np.linalg.det(S):.4f}')
print(f'rank     : {np.linalg.matrix_rank(S)}  (no es 2)')

## 5️⃣ SVD — la descomposición universal

**Singular Value Decomposition**: cualquier matriz `M (m,n)` se descompone como:

```
M = U · Σ · Vᵀ
```

- `U (m, m)` — vectores singulares izquierdos (ortonormales)
- `Σ (m, n)` — diagonal de **valores singulares** (decrecientes ≥ 0)
- `Vᵀ (n, n)` — vectores singulares derechos (ortonormales)

**Aplicaciones**: PCA, recomendadores (matriz factorization), compresión de imágenes, pseudo-inversa.

In [ ]:
M = rng.normal(0, 1, (6, 4))
U, s, Vt = np.linalg.svd(M, full_matrices=False)

print(f'M.shape  : {M.shape}')
print(f'U.shape  : {U.shape}')
print(f's        : {s.round(3)}  ← valores singulares decrecientes')
print(f'Vt.shape : {Vt.shape}')

# Reconstrucción: M = U @ diag(s) @ Vt
M_reconstruido = U @ np.diag(s) @ Vt
print(f'\nreconstrucción OK: {np.allclose(M, M_reconstruido)}')

## 6️⃣ Eigenvalores y eigenvectores

Para matriz cuadrada `A`, `A v = λ v` donde `λ` es eigenvalor y `v` eigenvector.

**Base conceptual de PCA**: los eigenvectores de la matriz de covarianza son las direcciones de máxima varianza.

In [ ]:
# Matriz de covarianza simulada (simétrica positiva semi-definida)
X = rng.normal(0, 1, (100, 3))
C = np.cov(X.T)   # (3, 3)

# Para matrices simétricas, usa eigh (más rápido, garantiza eigenvalores reales)
eigvals, eigvecs = np.linalg.eigh(C)
print(f'eigvalues (asc): {eigvals.round(4)}')
print(f'\neigvectors (columnas):')
print(eigvecs.round(3))

# La varianza total = suma de eigenvalores
print(f'\ntraza(C)        : {np.trace(C):.4f}')
print(f'sum(eigenvalues): {eigvals.sum():.4f}  ← igual')

## ✅ Checklist

- [ ] Uso `@` para mult matricial, `*` para elementwise
- [ ] NUNCA uso `inv(A) @ b`, siempre `solve(A, b)`
- [ ] Sé qué retorna SVD y verifico la reconstrucción
- [ ] Uso `eigh` para matrices simétricas
- [ ] Conozco `norm`, `det`, `rank`, `cond`

## 📝 Homework

Ver `README.md`. inv vs solve benchmark, regresión lineal cerrada, SVD, eigen de covarianza.

## 🔗 Referencias

- [`numpy.linalg`](https://numpy.org/doc/stable/reference/routines.linalg.html)
- [PEP 465 — `@` operator](https://peps.python.org/pep-0465/)

➡️ **Siguiente:** [021 — Aleatoriedad y semillas](../021-numpy-aleatoriedad-y-semillas/README.md)